## Model Deployment

## Learning Objectives

1. Explain the end-to-end workflow for deploying a trained machine learning model, including model loading, user interaction, prediction generation, and application hosting.
2. Build an interactive web application that accepts user input through widgets such as text fields, sliders, dropdowns, and file uploads.
3. Integrate a trained machine learning model into an application using framework-specific libraries and loaders.
4. Implement data preprocessing and feature transformation steps within a deployment pipeline to ensure consistency between training and inference environments.
5. Evaluate and improve the usability, performance, and reliability of a deployed machine learning application by handling errors and validating inputs.
6. Deploy and share a machine learning application locally or via cloud platforms.

**Goal:** Take a trained scikit-learn model and deploy it locally as an interactive web app using Streamlit.

**Prerequisites:** You know how to train a `LinearRegression` using `pd.get_dummies()` and `StandardScaler`. This lecture covers what comes after training.

---

## Part 1: Why Deploy? (The Four-Block Model)

### The Problem with Notebooks

A trained model in a notebook helps exactly one person — you. Every time you want a prediction, you have to:

1. Open the notebook
2. Edit a code cell with new values
3. Re-run the cell
4. Read the output

Nobody else can use it. And even you have to touch code every time.

### The Solution: A Web App

Instead of editing code, the user moves sliders and clicks buttons:

```
Jupyter Notebook                          Streamlit App
─────────────────                        ─────────────
model.predict([[1200, 3, 2]])            [slider] ── 1200 sqft
  → $245,000                             [slider] ── 3 bedrooms
                                         [slider] ── 2 bathrooms
                                              ↓
                                         model.predict([[1200, 3, 2]])
                                           → $245,000 displayed on screen
```

The model does the same thing. The difference is **how we interact with it**.

### The Four-Block Model

Every deployed ML app has the same four parts:

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Block 1     │    │  Block 2     │    │  Block 3     │    │  Block 4     │
│  Load Model  │ →  │  Get Input   │ →  │  Predict     │ →  │  Show Result │
│              │    │  from User   │    │              │    │              │
│  (runs once) │    │  (per click) │    │  (per click) │    │  (per click) │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

- **Block 1** loads the saved model file into memory
- **Block 2** collects input values from the user via sliders, dropdowns, etc.
- **Block 3** processes the inputs and calls `model.predict()`
- **Block 4** displays the prediction back to the user

The whole lecture is just us filling in these four blocks with Streamlit code.

---

## Part 2: Block 1 — Loading the Model

### What We're Loading

After training, you saved your model to a file:

In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import joblib
import pandas as pd

df = pd.DataFrame({
    "sqft": [1200, 2500, 1800, 3000, 950, 2200, 1600, 2800, 1400, 2000],
    "bedrooms": [2, 4, 3, 5, 1, 3, 3, 4, 2, 3],
    "bathrooms": [1, 3, 2, 4, 1, 2, 2, 3, 1, 2],
    "lot_size": [3000, 8000, 5000, 12000, 2000, 7000, 4500, 10000, 3500, 6000],
    "neighborhood": ["Downtown", "Suburbs", "Suburbs", "Waterfront",
                     "Downtown", "Rural", "Suburbs", "Waterfront",
                     "Downtown", "Rural"],
    "property_type": ["Condo", "Single Family", "Single Family", "Single Family",
                      "Condo", "Single Family", "Townhouse", "Single Family",
                      "Condo", "Townhouse"],
    "has_fireplace": ["No", "Yes", "No", "Yes", "No", "Yes", "No", "Yes", "No", "Yes"],
    "price": [202451, 469926, 339715, 657845, 143488,
              409488, 294688, 589512, 216958, 357138],
})

y = df["price"]
X = df.drop(columns=["price"])

# Convert binary column to 0/1
X["has_fireplace"] = X["has_fireplace"].map({"Yes": 1, "No": 0})

# Create dummy variables for categorical columns
X = pd.get_dummies(X, columns=["neighborhood", "property_type"])

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train and save model
model = LinearRegression()
model.fit(X_scaled, y)
joblib.dump(model, "model.pkl")

['model.pkl']

The `.pkl` file contains everything the model learned: the coefficients, intercept, and any other internal data.

### Loading in the App

Loading reverses the save:

In [2]:
import joblib

model = joblib.load("model.pkl")

model

LinearRegression()

After this line runs, `model` is a fully working `LinearRegression` with `coef_`, `intercept_`, and `predict()` ready.

### What Else Do We Need to Load?

During training you used `pd.get_dummies()` to create dummy variables. This changed your feature columns from something like:

```
sqft, bedrooms, bathrooms, neighborhood, property_type
```

into something like:

```
sqft, bedrooms, bathrooms, neighborhood_Downtown, neighborhood_Suburbs,
neighborhood_Rural, property_type_Condo, property_type_Single Family,
property_type_Townhouse
```

Your app needs to produce those exact columns. So we need to save the column names during training and load them in the app:

In [3]:
# During training — save the list of column names
columns = X.columns.tolist()
joblib.dump(columns, "columns.pkl")

['columns.pkl']

In [4]:
# In the app — load them back
columns = joblib.load("columns.pkl")

You may also need a fitted `StandardScaler`:

In [5]:
# During training
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, "scaler.pkl")

# In the app
scaler = joblib.load("scaler.pkl")

### Key Point

The model file contains the learned parameters. The column list tells us what features the model expects. The scaler tells us how to scale new data the same way we scaled the training data. **All three must be saved during training and loaded in the app.**

### The Files We'll Need

```
project/
├── app.py                 ← the Streamlit app (we will write this)
├── model.pkl              ← trained LinearRegression
├── training_columns.pkl   ← column names after pd.get_dummies()
└── scaler.pkl             ← fitted StandardScaler (if used)
```

---

## Part 3: Block 2 — Getting Input from the User

### How Streamlit Works

Streamlit is a Python library that turns Python scripts into web apps. Here is the most important thing to understand:

**A Streamlit script runs from top to bottom every time the user changes anything.**

No callbacks. No event listeners. Just variables.

```python
import streamlit as st

name = st.text_input("Your name")
st.write(f"Hello, {name}!")
```

When the user types in the text box, the whole script re-runs. `st.text_input` returns the current value, and `st.write` shows the greeting.

### The Four Widgets You Need

You only need four widgets for a house price predictor:

```python
# Numeric slider — for values in a range
sqft = st.slider("Square Footage", min_value=500, max_value=5000, value=1500)

# Precise number input — for values you type
lot_size = st.number_input("Lot Size (sqft)", min_value=1000, max_value=50000, value=5000)

# Dropdown — for picking one option from a list
neighborhood = st.selectbox("Neighborhood", ["Downtown", "Suburbs", "Rural", "Waterfront"])

# Checkbox — for yes/no features
has_fireplace = st.checkbox("Fireplace")
```

Each widget returns a Python variable:
- `st.slider` → an `int` or `float`
- `st.number_input` → an `int` or `float`
- `st.selectbox` → the selected string
- `st.checkbox` → `True` or `False`

### The Sidebar Pattern

Put your input widgets in the sidebar to keep the main area clean for results:

```python
st.title("Home Price Estimator")

with st.sidebar:
    st.header("Property Details")

    sqft = st.slider("Square Footage", 500, 5000, 1500)
    bedrooms = st.slider("Bedrooms", 1, 6, 3)
    bathrooms = st.slider("Bathrooms", 1, 5, 2)

    neighborhood = st.selectbox(
        "Neighborhood",
        ["Downtown", "Suburbs", "Rural", "Waterfront"]
    )

    has_fireplace = st.checkbox("Fireplace")
```

---

## Part 4: Block 3 — Preparing Data and Predicting

### The Big Challenge

During training, you did this:

```python
# Training
X = pd.get_dummies(X, columns=["neighborhood", "property_type"])
X_scaled = scaler.fit_transform(X)
model.fit(X_scaled, y)
```

The app must do the **exact same steps** before calling `model.predict()`. If the steps don't match, the predictions will be wrong — and the app won't tell you.

### Step 1: Create a DataFrame from Widget Values

```python
# Collect all widget values into a dictionary
input_data = {
    "sqft": sqft,                          # from st.slider
    "bedrooms": bedrooms,                  # from st.slider
    "bathrooms": bathrooms,                # from st.slider
    "neighborhood": neighborhood,          # from st.selectbox (string!)
    "has_fireplace": int(has_fireplace),   # from st.checkbox (convert to 0/1)
}

# Convert to a one-row DataFrame
input_df = pd.DataFrame([input_data])
```

### Step 2: Create Dummy Variables

This is the trickiest part. `pd.get_dummies()` on a single row only creates columns for categories that actually appear in that row.

Example: if the user selects `"Downtown"`, `pd.get_dummies()` creates one column:

```
neighborhood_Downtown  →  1
```

But the model was trained with these columns:

```
neighborhood_Downtown, neighborhood_Suburbs, neighborhood_Rural, neighborhood_Waterfront
```

**The app must produce all the same columns, with 0s for categories the user did not select.**

The pattern is:

```python
# Step 2a: Call get_dummies on the single row
input_encoded = pd.get_dummies(input_df)

# Step 2b: Add any missing columns, fill with 0
# training_columns was saved after pd.get_dummies() during training
input_encoded = input_encoded.reindex(columns=training_columns, fill_value=0)
```

What `reindex` does: it keeps only the columns in `training_columns`, adds any missing ones (filled with 0), and puts them in the same order.

### Step 3: Scale the Numeric Features

If you used `StandardScaler` during training, apply the **same** scaler:

```python
input_scaled = scaler.transform(input_encoded)
```

Notice: we call `.transform()`, not `.fit_transform()`. The scaler was already fitted during training. Calling `.fit()` in the app would compute new statistics from your single row, which is wrong.

### Step 4: Predict

```python
prediction = model.predict(input_scaled)[0]
```

`model.predict()` always returns an array (even for one sample). We use `[0]` to get the single value out.

### The Complete Block 3 Function

```python
def predict_price(sqft, bedrooms, bathrooms, neighborhood, has_fireplace):
    # Step 1: Build DataFrame
    input_data = {
        "sqft": sqft,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "neighborhood": neighborhood,
        "has_fireplace": int(has_fireplace),
    }
    input_df = pd.DataFrame([input_data])

    # Step 2: Create dummy variables matching training
    input_encoded = pd.get_dummies(input_df)
    input_encoded = input_encoded.reindex(columns=training_columns, fill_value=0)

    # Step 3: Scale
    input_scaled = scaler.transform(input_encoded)

    # Step 4: Predict
    prediction = model.predict(input_scaled)[0]

    return prediction
```

---

## Part 5: Block 4 — Displaying Results

### Showing the Prediction

The simplest display is the metric widget:

```python
st.metric("Estimated Price", f"${prediction:,.2f}")
```

The `f"${prediction:,.2f}"` formats the number with commas and two decimal places:
- `,` → adds commas (e.g., 245000 → 245,000)
- `.2f` → two decimal places

### The Empty State

What should the user see before they click anything? A simple message:

```python
st.info("Use the sidebar to enter property details and see an estimated price.")
```

---

## Part 6: The Complete App

### The Full Code

```python
import streamlit as st
import joblib
import pandas as pd

# ── App title (must be first Streamlit command) ──
st.set_page_config(page_title="Home Price Estimator", layout="centered")
st.title("Home Price Estimator")

# ── Block 1: Load model, scaler, and column names ──
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")
training_columns = joblib.load("columns.pkl")

# ── Block 2: Get input from user ──
with st.sidebar:
    st.header("Property Details")

    sqft = st.slider("Square Footage", 500, 5000, 1500)
    bedrooms = st.slider("Bedrooms", 1, 6, 3)
    bathrooms = st.slider("Bathrooms", 1, 5, 2)
    lot_size = st.number_input("Lot Size (sqft)", 1000, 50000, 5000)

    neighborhood = st.selectbox(
        "Neighborhood",
        ["Downtown", "Suburbs", "Rural", "Waterfront"]
    )

    has_fireplace = st.checkbox("Fireplace")

# ── Block 3: Prepare data and predict ──

# Step 1: Build DataFrame
input_data = {
    "sqft": sqft,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "lot_size": lot_size,
    "neighborhood": neighborhood,
    "has_fireplace": int(has_fireplace),
}
input_df = pd.DataFrame([input_data])

# Step 2: Create dummy variables matching training
input_encoded = pd.get_dummies(input_df)
input_encoded = input_encoded.reindex(columns=columns, fill_value=0)

# Step 3: Scale
input_scaled = scaler.transform(input_encoded)

# Step 4: Predict
prediction = model.predict(input_scaled)[0]

# ── Block 4: Display result ──
st.metric("Estimated Price", f"${prediction:,.2f}")
```

## Part 7: Running the App

### Step 1: Install Streamlit (If Not Installed)

```python
pip install streamlit
```

If you already have a `requirements.txt`, you can add it there.

### Step 2: Make Sure You Have All the Files

Check that your project folder contains:

```
project/
├── app.py                 ← the code above
├── model.pkl              ← your trained model
├── training_columns.pkl   ← column names from training
└── scaler.pkl             ← fitted StandardScaler
```

If any file is missing, the app will crash with a `FileNotFoundError`.

### Step 3: Run the App

```python
streamlit run app.py
```

A browser window should open at `http://localhost:8501` showing your app. Move a slider and watch the prediction update.

### If Something Goes Wrong

| Error | Likely Cause | Fix |
|---|---|---|
| `FileNotFoundError: model.pkl` | Missing model file | Make sure `model.pkl` is in the same folder |
| `ModuleNotFoundError: streamlit` | Streamlit not installed | `pip install streamlit` |
| `ValueError: X has X features, but LinearRegression is expecting X features` | Column mismatch | Check that `training_columns.pkl` matches the training data columns |
| Prediction is `NaN` or garbage | Scaling mismatch | Make sure you used `scaler.transform()`, not `scaler.fit_transform()` |

---

## Part 8: Basic Validation

### The Problem

What happens if the user enters 0 bedrooms? Or 10 bathrooms? The model may produce ridiculous predictions without any warning.

### Adding Input Checks

After collecting widget values and before predicting, add a validation step:

```python
# ── Validation ──
if bedrooms == 0:
    st.error("Bedrooms must be at least 1.")
    st.stop()  # Stop the app here — don't try to predict

prediction = model.predict(input_scaled)[0]
```

`st.stop()` halts the script. The error message shows, but the rest of the code doesn't run.

### Multiple Checks

You can check multiple conditions:

```python
errors = []

if bedrooms <= 0:
    errors.append("Bedrooms must be at least 1.")

if bathrooms > bedrooms:
    errors.append("Bathrooms cannot exceed bedrooms.")

if sqft <= 0:
    errors.append("Square footage must be positive.")

if errors:
    for error in errors:
        st.warning(error)
    st.stop()

# Only reaches here if all checks pass
prediction = model.predict(input_scaled)[0]
```

### Wrapping Predict in try/except

Sometimes the error comes from `model.predict()` itself:

```python
try:
    prediction = model.predict(input_scaled)[0]
except Exception as e:
    st.error(f"Prediction failed: {e}")
    st.stop()

# Only reaches here if prediction succeeded
st.metric("Estimated Price", f"${prediction:,.2f}")
```

---

## Recap: What You Learned

### The Four-Block Mental Model

You can now draw this diagram and explain what each block does:

```
Load Model → Get Input → Predict → Show Result
```

### Key Code Patterns

| What | Code |
|---|---|
| Load a model | `model = joblib.load("model.pkl")` |
| Numeric input | `st.slider("Label", min, max, default)` |
| Dropdown | `st.selectbox("Label", ["A", "B", "C"])` |
| Checkbox | `st.checkbox("Label")` — returns `True`/`False` |
| Create dummy columns | `pd.get_dummies(df)` then `.reindex(columns=..., fill_value=0)` |
| Scale new data | `scaler.transform(data)` — never `.fit_transform()` |
| Predict | `model.predict(data)[0]` |
| Display a value | `st.metric("Label", f"${value:,.2f}")` |
| Validate inputs | `if bad_condition: st.warning("..."); st.stop()` |
| Run the app | `streamlit run app.py` |

### What You Should Be Able to Do

1. Take a trained model (and scaler, and column list) from a Jupyter notebook
2. Write a Streamlit app that loads these files
3. Build a sidebar with sliders, dropdowns, and checkboxes for user input
4. Replicate the same preprocessing steps (`pd.get_dummies()` + scaling) that were used during training
5. Display the prediction and a breakdown of how it was calculated
6. Run the app locally with `streamlit run app.py`
7. Add basic validation to catch bad inputs

